# Entity Embedding

Goals:
* Learn how to train entity embeddings
* Learn entity embedding applications
* **Prove that depreciation patterns can be learned from age**


A common approach to learning representations for categorical variables is **entity embeddings** (EE), introduced by Entity Embeddings of Categorical Variables (2016). In this work, categorical values are mapped into a continuous vector space and learned jointly with the prediction task, allowing the model to place similar categories close together in the embedding space and improve generalization on sparse data.

This idea is closely related to earlier work in NLP, particularly **Efficient Estimation of Word Representations in Vector Space**, where words are embedded such that semantic similarity is captured by distance in the embedding space.

It is also conceptually closely related to representation learning methods developed in **speech processing** (Area that I am familiar with). In particular, **i-vectors** and later **x-vectors** (X-vectors: Robust DNN Embeddings for Speaker Recognition) learn fixed-dimensional embeddings that capture underlying structure (such as speaker identity) from high-dimensional inputs.

In this tutorial we use the craigslist_vehicles dataset from Kaggle. The data contains cars listed on the Craigslist listing service. Sellers can list their cars, add descriptions, mileage, photos, and set an asking price. Buyers can browse the list of available cars (online inventory) and contact the seller if they are interested in buying.

Our main goal is to learn how to train an Entity Embedding and prove that, despite age being numerical, we can treat it as categorical and create an Entity Embedding for age that learns a depreciation pattern.


ref:
* Entity Embeddings of Categorical Variables: https://arxiv.org/abs/1604.06737
* craigslist_vehicles dataset: https://www.kaggle.com/datasets/austinreese/craigslist-carstrucks-data
* X-vectors: Robust DNN Embeddings for Speaker Recognition: https://www.danielpovey.com/files/2018_icassp_xvectors.pdf

## Load and prepare enviroment

```sh
python3 -m venv ~/.venvs/entity-embedding-env

source ~/.venvs/entity-embedding-env/bin/activate

pip install --upgrade pip

pip install \
tensorflow \
tensorflow-datasets \
tensorflow-recommenders \
jupyter \
pandas \
matplotlib \
plotly 

pip install tensorflow tensorflow-datasets tensorflow-recommenders

pip install importlib-resources

# NOTE: optional if you are using vs code
pip install ipykernel

# NOTE: optional if you are using vs code
python -m ipykernel install \
--user \
--name entity-embedding \
--display-name "Python (entity-embedding)"

pip install scikit-learn

```

In [1]:
import IPython
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
import os
# NOTE: tensorflow_recommenders still use keras 2 but tensorflow switch to keras 3
# setting keras 2 as keras version for tensorflow env
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import tensorflow_datasets as tfds
import tensorflow_recommenders as tfrs

print("TF:", tf.__version__)
print("TFDS:", tfds.__version__)
print("TFRS:", tfrs.__version__)

TF: 2.16.2
TFDS: 4.9.9
TFRS: v0.7.7


## Load dataset: Used Cars Dataset 

In [3]:
import pandas as pd

import kagglehub
from kagglehub import KaggleDatasetAdapter

_load_fom_kaggle = False

local_file_name = "craigslist_vehicles.csv"

if _load_fom_kaggle:

    craigslist_vehicles = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "mbaabuharun/craigslist-vehicles",
        local_file_name,
    )
    # caching the file locally for future use as pkl
    craigslist_vehicles.to_csv(local_file_name, index=False)
    craigslist_vehicles.to_pickle(local_file_name.replace(".csv", ".pkl"))

else: 
    !pwd
    !ls *.csv
    #craigslist_vehicles = pd.read_csv(local_file_name)
    craigslist_vehicles = pd.read_pickle(local_file_name.replace(".csv", ".pkl"))

print(f"local file name: {local_file_name}")
print(craigslist_vehicles.shape)
craigslist_vehicles.head()



/Users/leandro.fernandes/leandro/machine_learning_algorithms
craigslist_vehicles.csv
local file name: craigslist_vehicles.csv
(426880, 28)


,Unnamed: 0,id,url,region,region_url,price,year,manufacturer,model,condition,...,type,paint_color,image_url,description,county,state,lat,long,posting_date,removal_date
0,362773,7307679724,https://abilene.craigslist.org/ctd/d/abilene-2...,abilene,https://abilene.craigslist.org,4500,2002.0,bmw,x5,NaN,...,NaN,NaN,https://images.craigslist.org/00m0m_iba78h8ty9...,"$4,500 Cash 2002 BMW X5 8 cylinder 4.4L moto...",NaN,tx,32.401556,-99.884713,2021-04-16 00:00:00+00:00,2021-05-02 00:00:00+00:00
1,362712,7311833696,https://abilene.craigslist.org/ctd/d/abilene-2...,abilene,https://abilene.craigslist.org,4500,2002.0,bmw,x5,NaN,...,NaN,NaN,https://images.craigslist.org/00m0m_iba78h8ty9...,"$4,500 Cash 2002 BMW X5 8 cylinder 4.4L moto...",NaN,tx,32.401556,-99.884713,2021-04-24 00:00:00+00:00,2021-04-28 00:00:00+00:00
2,362722,7311441996,https://abilene.craigslist.org/ctd/d/abilene-2...,abilene,https://abilene.craigslist.org,4900,2006.0,toyota,camry,excellent,...,sedan,silver,https://images.craigslist.org/00808_5FkOw2aGjA...,2006 TOYOTA CAMRY LE Sedan Ready To Upgrade ...,NaN,tx,32.453848,-99.787900,2021-04-23 00:00:00+00:00,2021-05-25 00:00:00+00:00
3,362771,7307680715,https://abilene.craigslist.org/ctd/d/abilene-2...,abilene,https://abilene.craigslist.org,6500,2008.0,ford,expedition,NaN,...,NaN,NaN,https://images.craigslist.org/00M0M_i9CoFvVq8o...,$6500.00 2008 Ford Expedition 8 cylinder 5.4L...,NaN,tx,32.401556,-99.884713,2021-04-16 00:00:00+00:00,2021-04-26 00:00:00+00:00
4,362710,7311834578,https://abilene.craigslist.org/ctd/d/abilene-2...,abilene,https://abilene.craigslist.org,6500,2008.0,ford,expedition,NaN,...,NaN,NaN,https://images.craigslist.org/00M0M_i9CoFvVq8o...,$6500.00 2008 Ford Expedition 8 cylinder 5.4L...,NaN,tx,32.401556,-99.884713,2021-04-24 00:00:00+00:00,2021-05-12 00:00:00+00:00


In [4]:
craigslist_vehicles.shape

craigslist_vehicles.columns.tolist()
craigslist_vehicles.sample(11)

(426880, 28)

['Unnamed: 0',
 'id',
 'url',
 'region',
 'region_url',
 'price',
 'year',
 'manufacturer',
 'model',
 'condition',
 'cylinders',
 'fuel',
 'odometer',
 'title_status',
 'transmission',
 'VIN',
 'drive',
 'size',
 'type',
 'paint_color',
 'image_url',
 'description',
 'county',
 'state',
 'lat',
 'long',
 'posting_date',
 'removal_date']

,Unnamed: 0,id,url,region,region_url,price,year,manufacturer,model,condition,...,type,paint_color,image_url,description,county,state,lat,long,posting_date,removal_date
83489,296633,7310893992,https://columbus.craigslist.org/cto/d/columbus...,columbus,https://columbus.craigslist.org,7000,2009.0,ford,mustang,excellent,...,coupe,silver,https://images.craigslist.org/00z0z_9jI5jkItnl...,"Price reduced to 7,000 from 8,000 2009 Ford M...",NaN,oh,40.099900,-83.015700,2021-04-22 00:00:00+00:00,2021-05-11 00:00:00+00:00
95899,92790,7310518021,https://daytona.craigslist.org/ctd/d/orlando-2...,daytona beach,https://daytona.craigslist.org,23900,2013.0,dodge,grand caravan rt,excellent,...,van,NaN,https://images.craigslist.org/00v0v_g5pbedAhQt...,2013 Dodge Grand Caravan *Wheelchair Van* *Han...,NaN,fl,28.596980,-81.398555,2021-04-21 00:00:00+00:00,2021-04-29 00:00:00+00:00
276070,161320,7313302541,https://omaha.craigslist.org/ctd/d/omaha-2020-...,omaha / council bluffs,https://omaha.craigslist.org,17995,NaN,NaN,Renegade,NaN,...,SUV,grey,https://images.craigslist.org/00x0x_aXDYO0lT3s...,"2020 *Jeep* *Renegade* Altitude FWD SUV - $17,...",NaN,ia,41.207382,-96.023096,2021-04-27 00:00:00+00:00,2021-05-06 00:00:00+00:00
143774,200371,7315779356,https://grandrapids.craigslist.org/cto/d/trave...,grand rapids,https://grandrapids.craigslist.org,21000,2012.0,NaN,ISUZU NPR-HD Box Truck,like new,...,truck,white,https://images.craigslist.org/00U0U_2Zy4BSUush...,FOR SALE TRUCK WITH OR WITHOUT EQUIPMENT 16 Ft...,NaN,mi,44.764262,-85.613279,2021-05-02 00:00:00+00:00,2021-05-11 00:00:00+00:00
41905,392636,7310376747,https://blacksburg.craigslist.org/ctd/d/radfor...,new river valley,https://blacksburg.craigslist.org,15590,2017.0,kia,forte lx sedan 4d,good,...,sedan,white,https://images.craigslist.org/00N0N_1xMPvfxRAI...,Carvana is the safer way to buy a car During t...,NaN,va,37.120000,-80.550000,2021-04-21 00:00:00+00:00,2021-04-28 00:00:00+00:00
43993,133294,7312761505,https://boise.craigslist.org/ctd/d/boise-2020-...,boise,https://boise.craigslist.org,20504,2020.0,kia,optima,NaN,...,sedan,silver,https://images.craigslist.org/00x0x_iz0zBsENgL...,Dennis Dillon Mazda Kia address: 9501 W Fai...,NaN,id,43.618712,-116.300339,2021-04-26 00:00:00+00:00,2021-05-07 00:00:00+00:00
82435,295817,7314438077,https://columbus.craigslist.org/ctd/d/lake-in-...,columbus,https://columbus.craigslist.org,21900,2015.0,chevrolet,g4500,excellent,...,bus,white,https://images.craigslist.org/00z0z_lhyYbxmfMo...,Signature Truck CenterAsk for: Craigslist Sale...,NaN,oh,42.196156,-88.310560,2021-04-29 00:00:00+00:00,2021-05-17 00:00:00+00:00
339231,58804,7315549552,https://santabarbara.craigslist.org/cto/d/buel...,santa barbara,https://santabarbara.craigslist.org,25000,1959.0,NaN,willys,excellent,...,NaN,NaN,https://images.craigslist.org/00Q0Q_gnCelO5ZY8...,1959 Willys wagon. Great running Chevy 350 wit...,NaN,ca,34.648800,-120.170100,2021-05-01 00:00:00+00:00,2021-05-09 00:00:00+00:00
35576,311581,7314651593,https://bend.craigslist.org/ctd/d/gladstone-20...,bend,https://bend.craigslist.org,11985,2006.0,honda,pilot,excellent,...,SUV,grey,https://images.craigslist.org/01313_dba0iHoXBW...,2006 Honda Pilot EX L w/DVD 4dr SUV 4WD SUV ...,NaN,or,45.374861,-122.602289,2021-04-30 00:00:00+00:00,2021-05-18 00:00:00+00:00
173801,98090,7316318370,https://jacksonville.craigslist.org/ctd/d/jack...,jacksonville,https://jacksonville.craigslist.org,0,2014.0,nissan,altima,excellent,...,sedan,black,https://images.craigslist.org/00d0d_2Gzh6oYYo0...,Mention that you heard about this vehicle on C...,NaN,fl,30.318329,-81.730757,2021-05-03 00:00:00+00:00,2021-05-13 00:00:00+00:00


In [5]:
craigslist_vehicles

,Unnamed: 0,id,url,region,region_url,price,year,manufacturer,model,condition,...,type,paint_color,image_url,description,county,state,lat,long,posting_date,removal_date
0,362773,7307679724,https://abilene.craigslist.org/ctd/d/abilene-2...,abilene,https://abilene.craigslist.org,4500,2002.0,bmw,x5,NaN,...,NaN,NaN,https://images.craigslist.org/00m0m_iba78h8ty9...,"$4,500 Cash 2002 BMW X5 8 cylinder 4.4L moto...",NaN,tx,32.401556,-99.884713,2021-04-16 00:00:00+00:00,2021-05-02 00:00:00+00:00
1,362712,7311833696,https://abilene.craigslist.org/ctd/d/abilene-2...,abilene,https://abilene.craigslist.org,4500,2002.0,bmw,x5,NaN,...,NaN,NaN,https://images.craigslist.org/00m0m_iba78h8ty9...,"$4,500 Cash 2002 BMW X5 8 cylinder 4.4L moto...",NaN,tx,32.401556,-99.884713,2021-04-24 00:00:00+00:00,2021-04-28 00:00:00+00:00
2,362722,7311441996,https://abilene.craigslist.org/ctd/d/abilene-2...,abilene,https://abilene.craigslist.org,4900,2006.0,toyota,camry,excellent,...,sedan,silver,https://images.craigslist.org/00808_5FkOw2aGjA...,2006 TOYOTA CAMRY LE Sedan Ready To Upgrade ...,NaN,tx,32.453848,-99.787900,2021-04-23 00:00:00+00:00,2021-05-25 00:00:00+00:00
3,362771,7307680715,https://abilene.craigslist.org/ctd/d/abilene-2...,abilene,https://abilene.craigslist.org,6500,2008.0,ford,expedition,NaN,...,NaN,NaN,https://images.craigslist.org/00M0M_i9CoFvVq8o...,$6500.00 2008 Ford Expedition 8 cylinder 5.4L...,NaN,tx,32.401556,-99.884713,2021-04-16 00:00:00+00:00,2021-04-26 00:00:00+00:00
4,362710,7311834578,https://abilene.craigslist.org/ctd/d/abilene-2...,abilene,https://abilene.craigslist.org,6500,2008.0,ford,expedition,NaN,...,NaN,NaN,https://images.craigslist.org/00M0M_i9CoFvVq8o...,$6500.00 2008 Ford Expedition 8 cylinder 5.4L...,NaN,tx,32.401556,-99.884713,2021-04-24 00:00:00+00:00,2021-05-12 00:00:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
426875,303849,7307070484,https://zanesville.craigslist.org/cto/d/zanesv...,zanesville / cambridge,https://zanesville.craigslist.org,5100,2009.0,NaN,saab 9-7x,fair,...,SUV,grey,https://images.craigslist.org/00b0b_dsIhheG86S...,For sale: 2009 Saab 9-7x Fair condition AWD ...,NaN,oh,39.937000,-82.031500,2021-04-15 00:00:00+00:00,2021-04-21 00:00:00+00:00
426876,303706,7314635557,https://zanesville.craigslist.org/cto/d/zanesv...,zanesville / cambridge,https://zanesville.craigslist.org,7500,2011.0,ford,f-450,good,...,bus,red,https://images.craigslist.org/00b0b_8lBfNkZ6pr...,"2011 E-Ford 450 with 177k miles, 6.8 liter v-1...",NaN,oh,39.927400,-82.004100,2021-04-30 00:00:00+00:00,2021-05-15 00:00:00+00:00
426877,303704,7314710341,https://zanesville.craigslist.org/cto/d/zanesv...,zanesville / cambridge,https://zanesville.craigslist.org,25000,2016.0,chevrolet,silverado,excellent,...,truck,blue,https://images.craigslist.org/00k0k_jw0Pda6LTk...,2013 Silverado excellent condition. Blue in co...,NaN,oh,39.896865,-82.042283,2021-04-30 00:00:00+00:00,2021-05-12 00:00:00+00:00
426878,303670,7316225330,https://zanesville.craigslist.org/cto/d/zanesv...,zanesville / cambridge,https://zanesville.craigslist.org,6,1986.0,NaN,camaro iroc z28,good,...,NaN,red,https://images.craigslist.org/00Y0Y_avlrYDn7OY...,"1986 Iroc Z28 , T-tops , 86,500 miles , has ne...",NaN,oh,39.938630,-82.006760,2021-05-03 00:00:00+00:00,2021-05-08 00:00:00+00:00


## Preprocessing

In [6]:
import numpy as np
from pandas.api.types import is_datetime64_any_dtype

def requires(criteria, message: str):

    if not criteria:
        raise ValueError(message)

    
def preprocess_data(raw_data: pd.DataFrame) -> pd.DataFrame:

    cols = [
        "price",
        "year",
        "posting_date",
        "type",
        "manufacturer",
        "model",
        "fuel",
        "transmission",
        "condition",
        "drive",
        "odometer"
    ]

    preprocessed_data = raw_data[cols].copy()
    preprocessed_data = preprocessed_data.dropna()
    preprocessed_data["posting_date"] = pd.to_datetime(preprocessed_data["posting_date"], errors="coerce")

    #preprocessed_data["posting_date"] = (pd.to_datetime(preprocessed_data["posting_date"], errors="coerce").dt.date)

    preprocessed_data["year"] = preprocessed_data["year"].astype(int)
    preprocessed_data["price"] = preprocessed_data["price"].astype(int)
    preprocessed_data["odometer"] = preprocessed_data["odometer"].astype(int)
    preprocessed_data["odometer"] = pd.to_numeric(preprocessed_data["odometer"],errors="coerce")

    return preprocessed_data


def filter_data(data: pd.DataFrame) -> pd.DataFrame:

    data = data[data["price"] > 1_000]
    data = data[data["price"] < 100_000]
    data = data[data["year"] > 1990]
    data = data[data["year"] < 2024]

    data = data[(data["odometer"] > 0) & (data["odometer"] < 500000)]

    return data


def _get_age(data: pd.DataFrame) -> pd.Series:

    age = (data["posting_date"].dt.year - data["year"]).astype("int")

    return age

def _reduce_taxonomy_cardinality(data: pd.DataFrame) -> pd.DataFrame:

    # Reduce the cardinality of the "model" column by keeping only the top 25 most frequent models
    _n_top_model = 750
    top_models = data["model"].value_counts().nlargest(_n_top_model).index
    data["model"] = np.where(data["model"].isin(top_models), data["model"], "other")

    _n_top_manufacturer = 50
    top_manufacturers = data["manufacturer"].value_counts().nlargest(_n_top_manufacturer).index
    data["manufacturer"] = np.where(data["manufacturer"].isin(top_manufacturers), data["manufacturer"], "other")

    _n_top_type = 10 
    top_type = data["type"].value_counts().nlargest(_n_top_type).index
    data["type"] = np.where(data["type"].isin(top_type), data["type"], "other")

    return data

def _get_age_as_category(age: pd.Series) -> pd.Series:

    categorical_age = age.astype(int).astype(str)
    categorical_age = np.where(age > 19, "old", categorical_age)
    
    return categorical_age

def _normalize_text_columns(data: pd.DataFrame, col: str) -> pd.DataFrame:
    
    data[col] = (
        data[col]
        .fillna("unknown")
        .astype(str)
        .str.lower()
        .str.strip()
    )

    # removing special chars
    special_chars = [' ', '-', '/', '(', ')', '.']
    for char in special_chars:
        data[col] = data[col].str.replace(char, '', regex=False)

    return data

def _normalize_condition_column(condition: pd.Series) -> pd.Series:

    condition_map = {
        "new": "excellent",
        "likenew": "excellent",
        "excellent": "excellent",

        "good": "medium",

        "fair": "low",
        "salvage": "low"
    }

    normalized_condition = condition.map(condition_map).fillna("unknown")

    return normalized_condition

def _get_usage_level(data: pd.DataFrame) -> pd.Series:
    data_view = data[["odometer", "age"]].copy()

    data_view["age_safe"] = data_view["age"].clip(lower=-1)
    data_view["odometer_per_age_buckets"] = data_view["odometer"] / (data_view["age_safe"] + 2)

    _percentiles = (
        data_view.groupby("age")["odometer_per_age_buckets"]
        .quantile([0.16, 0.33, 0.50, 0.66, 0.82])
        .unstack()
        .rename(columns={0.16: "p16", 0.33: "p33", 0.50: "p50", 0.66: "p66", 0.82: "p82"})
    )

    data_view = data_view.join(_percentiles, on="age")

    def _usage_bucket(row):
        if pd.isna(row["odometer_per_age_buckets"]) or pd.isna(row["p16"]):
            return "unknown"

        v = row["odometer_per_age_buckets"]

        if v <= row["p16"]:
            return "very_low"
        elif v <= row["p33"]:
            return "low"
        elif v <= row["p50"]:
            return "medium"
        elif v <= row["p66"]:
            return "high"
        elif v <= row["p82"]:
            return "very_high"
        return "extreme"

    usage_level = data_view.apply(_usage_bucket, axis=1)

    return usage_level.fillna("unknown")

def _get_time_features(data: pd.Series) -> pd.Series:

    requires("posting_date" in data.columns, "posting_date column is required to extract time features")
    requires(
        is_datetime64_any_dtype(data["posting_date"]),
        f"posting_date column must be datetime type, got: {data['posting_date'].dtype}"
    )

    posting_date = data["posting_date"]

    data["posting_day"] = posting_date.dt.day.astype(str)

    data["day_of_week"] = posting_date.dt.weekday.astype(str)

    # NOTE: skip bc redundant with day of week and posting day
    # data["is_weekend"] = (
    #     data["day_of_week"] >= 5
    # ).astype(int)

    data["posting_month"] = posting_date.dt.month.astype(str)
    # NOTE: not useful. datset is only 2 month of data
    # data["posting_quarter"] = posting_date.dt.quarter

    # data["posting_semester"] = np.where(
    #     posting_date.dt.month <= 6,
    #     1,
    #     2
    # )
    # data["posting_year"] = posting_date.dt.year

    return data

categorical_cols = [
    "type",
    "manufacturer",
    "model",
    "fuel",
    "transmission",
    "condition",
    "drive",
    "age_category",
    "usage_level",
    "posting_month",
    "posting_day",
    "day_of_week",
    # "is_weekend"
]


numerical_cols = [
    "year",
    "odometer",
    "age",
    # "odometer_per_age_buckets"
]


def feature_engineering(preprocessed_data: pd.DataFrame) -> pd.DataFrame:

    data_engineered = preprocessed_data.copy()

    for col in ["type", "manufacturer", "model", "fuel", "transmission", "condition", "drive"]:
        data_engineered = _normalize_text_columns(data_engineered, col)

    data_engineered = _reduce_taxonomy_cardinality(data_engineered)

    data_engineered["age"] = _get_age(data_engineered)
    data_engineered["age_category"] = _get_age_as_category(data_engineered["age"])

    data_engineered["usage_level"] = _get_usage_level(data_engineered)

    data_engineered["condition"] = _normalize_condition_column(data_engineered["condition"])

    data_engineered = _get_time_features(data_engineered)

    return data_engineered


In [7]:
preprocessed_data = preprocess_data(craigslist_vehicles)
filtered_data = filter_data(preprocessed_data)
featured_data = feature_engineering(filtered_data)

craigslist_vehicles.shape
featured_data.shape
featured_data.head(5)

(426880, 28)

(148421, 17)

,price,year,posting_date,type,manufacturer,model,fuel,transmission,condition,drive,odometer,age,age_category,usage_level,posting_day,day_of_week,posting_month
2,4900,2006,2021-04-23 00:00:00+00:00,sedan,toyota,camry,gas,automatic,excellent,fwd,184930,15,15,very_high,23,4,4
7,11500,2014,2021-04-23 00:00:00+00:00,suv,honda,crv,gas,automatic,excellent,fwd,151299,7,7,extreme,23,4,4
8,11125,2014,2021-04-18 00:00:00+00:00,sedan,toyota,camry,gas,automatic,medium,fwd,51534,7,7,low,18,6,4
9,13500,2016,2021-04-20 00:00:00+00:00,other,chrysler,town&country,gas,automatic,excellent,fwd,107699,5,5,extreme,20,1,4
10,11850,2016,2021-04-12 00:00:00+00:00,sedan,toyota,corolla,gas,automatic,excellent,fwd,30323,5,5,low,12,0,4


## EDA

In [8]:
# NOTE: inspecting transmission and type values
featured_data.transmission.unique()
craigslist_vehicles.transmission.unique()

craigslist_vehicles.type.unique()

craigslist_vehicles.type.fillna("unknow").value_counts(normalize=True)


featured_data.groupby("transmission")["price"].describe()
featured_data.groupby("transmission")["age"].describe()


<ArrowStringArray>
['automatic', 'manual', 'other']
Length: 3, dtype: str

<ArrowStringArray>
['automatic', 'other', 'manual', nan]
Length: 4, dtype: str

<ArrowStringArray>
[          nan,       'sedan',         'SUV',    'mini-van',      'pickup',
       'truck',       'coupe',   'hatchback', 'convertible',       'other',
       'wagon',         'van',     'offroad',         'bus']
Length: 14, dtype: str

type
unknow         0.217527
sedan          0.203936
SUV            0.181044
pickup         0.101926
truck          0.082644
other          0.051794
coupe          0.044987
hatchback      0.038882
wagon          0.025185
van            0.020024
convertible    0.018110
mini-van       0.011303
offroad        0.001427
bus            0.001211
Name: proportion, dtype: float64

,count,mean,std,min,25%,50%,75%,max
transmission,,,,,,,,
automatic,109126.0,15050.818879,12124.058105,1012.0,6499.0,11300.0,19990.0,99999.0
manual,7438.0,13029.984136,11324.564760,1100.0,5000.0,9000.0,17995.0,92995.0
other,31857.0,28418.283391,8804.086746,1199.0,20990.0,28590.0,34990.0,85867.0


,count,mean,std,min,25%,50%,75%,max
transmission,,,,,,,,
automatic,109126.0,10.218848,5.594207,-1.0,6.0,9.0,14.0,30.0
manual,7438.0,13.827373,6.748108,0.0,8.0,13.0,18.0,30.0
other,31857.0,4.191135,2.842792,0.0,2.0,4.0,5.0,30.0


In [9]:
craigslist_vehicles.model.value_counts(normalize=False).head(500).tail(25)

model
transit van                   135
ion                           134
soul +                        134
Scion xD Hatchback 4D         134
c-class c 300                 134
tahoe ltz                     134
a6 2.0t premium sedan 4d      134
highlander limited            134
cooper countryman             134
1500 sport 4x4 1/2 ton        134
silverado 1500 ltz            133
s60 t6 r-design sedan 4d      133
acadia sle-2 sport utility    132
silverado 2500 hd             132
sonata sel sedan 4d           132
7 series                      132
cooper hardtop                132
glc                           132
xf 20d premium sedan 4d       131
Genesis G70 2.0T Sedan 4D     131
ilx                           131
camry hybrid                  131
f150 supercrew cab fx4        130
cts 2.0 luxury sedan 4d       130
continental reserve           130
Name: count, dtype: int64

In [10]:
# NOTE: inspecting price values impact by time features
featured_data.groupby("posting_month")["price"].describe()
# featured_data.groupby("posting_day")["price"].describe()
featured_data.groupby("day_of_week")["price"].describe()

,count,mean,std,min,25%,50%,75%,max
posting_month,,,,,,,,
4,107141.0,18336.945586,12856.568175,1012.0,7900.0,15498.0,26990.0,98995.0
5,41280.0,16473.722141,12291.604583,1012.0,6750.0,13500.0,23990.0,99999.0


,count,mean,std,min,25%,50%,75%,max
day_of_week,,,,,,,,
0,26811.0,17700.265786,12645.493653,1100.0,7500.0,14990.0,25590.0,99990.0
1,27251.0,17023.015229,12424.917636,1012.0,7000.0,13999.0,23995.0,98995.0
2,16741.0,18312.299086,12911.657355,1012.0,7950.0,15590.0,26590.0,99999.0
3,16874.0,17238.194323,12463.421094,1029.0,7300.0,14000.0,23995.0,98900.0
4,22750.0,18301.915121,13053.521229,1055.0,7600.0,14995.0,27025.0,97500.0
5,22565.0,17849.218702,12567.454169,1100.0,7500.0,14995.0,25990.0,98900.0
6,15429.0,18772.327695,13122.742404,1100.0,6999.0,16590.0,28990.0,99700.0


In [11]:
# NOTE: inspecting price values impact by age and usage level features (LLM help with this code)
usage_order = [
    "very_low",
    "low",
    "medium",
    "high",
    "very_high",
    "extreme"
]

median_tbl = (
    featured_data.groupby(
        ["age", "usage_level"]
    )["price"]
    .median()
    .unstack()
    .reindex(columns=usage_order)
)

count_tbl = (
    featured_data.groupby(
        ["age", "usage_level"]
    )["price"]
    .size()
    .unstack()
    .reindex(columns=usage_order)
)

combined = (
    median_tbl.round(0).astype(str) +
    " (n=" + count_tbl.astype(str) + ")"
)

combined.head(11)

usage_level,very_low,low,medium,high,very_high,extreme
age,,,,,,
-1,27725.0 (n=1.0),NaN,30850.0 (n=1.0),NaN,4790.0 (n=1.0),3500.0 (n=1.0)
0,41604.0 (n=58.0),50345.0 (n=49.0),52498.0 (n=59.0),35990.0 (n=50.0),41945.0 (n=49.0),35785.0 (n=59.0)
1,35990.0 (n=1062.0),31590.0 (n=1090.0),33900.0 (n=1109.0),37590.0 (n=1043.0),34990.0 (n=1053.0),30590.0 (n=1165.0)
2,30990.0 (n=1286.0),35590.0 (n=1361.0),33995.0 (n=1379.0),33488.0 (n=1295.0),28990.0 (n=1264.0),26990.0 (n=1437.0)
3,28590.0 (n=1837.0),26990.0 (n=1936.0),28590.0 (n=1930.0),28990.0 (n=1825.0),25887.0 (n=1824.0),23999.0 (n=2053.0)
4,28990.0 (n=1961.0),25590.0 (n=1993.0),27590.0 (n=2030.0),23990.0 (n=1915.0),24995.0 (n=1914.0),20975.0 (n=2155.0)
5,29990.0 (n=1798.0),25590.0 (n=1561.0),21990.0 (n=1730.0),19995.0 (n=1632.0),16900.0 (n=1625.0),18500.0 (n=1832.0)
6,21590.0 (n=1613.0),19990.0 (n=1698.0),17592.0 (n=1704.0),16590.0 (n=1604.0),14998.0 (n=1604.0),17485.0 (n=1806.0)
7,23990.0 (n=1616.0),18990.0 (n=1695.0),15900.0 (n=1707.0),15000.0 (n=1605.0),13995.0 (n=1606.0),12900.0 (n=1805.0)


## Train and test split

In [12]:
from sklearn.model_selection import train_test_split

target_col = "price"

X = featured_data.drop(columns=[target_col]).copy()
y = featured_data[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

X_train.shape
X_test.shape

# NOTE: full dataframe help erros analyses and eda
train_data = X_train.copy()
train_data[target_col] = y_train

test_data = X_test.copy()
test_data[target_col] = y_test

(118736, 16)

(29685, 16)

## Define DNN

In this section we define the Entoty Embedding layers and entrire DNN.

**NOTES:**

1.  StringLookup(): similar to LabelEncoder. Motivations:

    * **handles unknown categories automatically**
        * Unknown/un seen examples are handle by the tag [UNK] and it represents of the average of unknow behavour
    * stays inside the model
    * saves vocabulary with the model
    * avoids train/serve mismatch

1. .adapt() is similar to fit(), tranform()

1.  Mental model

```
StringLookup:
    strings -> integers

Embedding:
    integers -> vectors

"ford"
  |
  v
StringLookup
  |
  v
3
  |
  v
Embedding
  |
  v
[0.12, -0.88, 0.45, ...]
```

 


```python
cat_cols = ["manufacturer", "fuel", "transmission", "day_of_week"]

inputs = {}
encoded_cat_layers = []

for col in cat_cols:
    inp = tf.keras.Input(shape=(1,), name=col, dtype=tf.string)
    lookup = tf.keras.layers.StringLookup()
    lookup.adapt(X_train[col])

    idx = lookup(inp)

    emb = tf.keras.layers.Embedding(
        input_dim=lookup.vocabulary_size(),
        output_dim=4,   # choose per feature
        name=f"{col}_emb",
    )(idx)

    vec = tf.keras.layers.Flatten()(emb)

    inputs[col] = inp
    encoded_cat_layers.append(vec)
```

numericals

```python
cat_cols = ["manufacturer", "fuel", "transmission", "day_of_week"]

inputs = {}
encoded_cat_layers = []

for col in cat_cols:
    inp = tf.keras.Input(shape=(1,), name=col, dtype=tf.string)
    lookup = tf.keras.layers.StringLookup()
    lookup.adapt(X_train[col])

    idx = lookup(inp)

    emb = tf.keras.layers.Embedding(
        input_dim=lookup.vocabulary_size(),
        output_dim=4,   # choose per feature
        name=f"{col}_emb",
    )(idx)

    vec = tf.keras.layers.Flatten()(emb)

    inputs[col] = inp
    encoded_cat_layers.append(vec)
```

In [13]:
import math

class EmbbedingBlock(tf.keras.Model):

    def __init__(self, categorical_feature_vocabs: dict[str, list[str]]) -> None:

        super().__init__()

        requires(
            len(categorical_feature_vocabs.keys()) > 0,
            "categorical_feature_vocabs must contain at least one feature"
        )

        self.feature_vocab = categorical_feature_vocabs
        self.features = list(categorical_feature_vocabs.keys())
        self.embeddings = {}
        self.encoders = {}

        self.build_embedding_and_encoders()

    def build_embedding_and_encoders(self):

        # NOTE: define the encoders and embeddings layers and dim for each categorical feature
        # feature_vocab = {c1: vocab, c2: vocab, ...}
        for feature in self.features:

            vocab = self.feature_vocab[feature]

            requires(
                len(vocab) > 1,
                f"vocab for feature {feature} must contain at least two values"
            )

            # encoders
            self.encoders[feature] = tf.keras.layers.StringLookup(
                vocabulary=vocab,
                mask_token=None
            )

            self.embeddings[feature] = tf.keras.layers.Embedding(
                input_dim=len(vocab) + 1,
                output_dim=min(int(math.sqrt(len(vocab))), 100),
                name=f"{feature}_embedding"
            )

    def call(self, X_cat):
        embeddings = []

        for feature in self.features:

            encoder = self.encoders[feature]
            feature_idx = encoder(X_cat[feature])
            feature_emb = self.embeddings[feature](feature_idx)
            feature_emb = tf.keras.layers.Flatten()(feature_emb)
            embeddings.append(feature_emb)

        x = tf.keras.layers.Concatenate()(embeddings)
        
        return x
    
    def get_embedding_matrix(self, feature: str):

        requires(
            feature in self.features,
            f"{feature} not found"
        )

        vocab = self.encoders[feature].get_vocabulary()

        matrix = self.embeddings[feature].get_weights()[0]


        ee_dict = {k:v for k, v in zip(vocab, matrix)}


        return ee_dict

class NumericalBlock(tf.keras.Model):
    
    def __init__(self, numerical_features: list[str]) -> None:

        super().__init__()
        requires(
            len(numerical_features) > 0,
            "numerical_features must contain at least one feature"
        )

        self.features = numerical_features

        self.numerical_inputs = {}
        self.numerical_layers = []

        self.build_numerical_inputs_layers()

    def build_numerical_inputs_layers(self):

        for c in self.features:
            input_layer = tf.keras.Input(shape=(1,), name=c, dtype=tf.float32)
            self.numerical_inputs[c] = input_layer

            self.numerical_layers.append(input_layer)

    def call(self, X_num):

        layers = [tf.cast(X_num[c], tf.float32) for c in self.features]

        return tf.keras.layers.Concatenate()(layers)

class DNNModel(tf.keras.Model):

    def __init__(self, categorical_feature_vocabs: dict[str, list[str]], numerical_features: list[str]) -> None:

        super().__init__()

        self.embedding_block = EmbbedingBlock(categorical_feature_vocabs)
        self.numerical_block = NumericalBlock(numerical_features)

        self.dense1 = tf.keras.layers.Dense(128, activation="relu")
        self.dense2 = tf.keras.layers.Dense(64, activation="relu")
        self.dense3 = tf.keras.layers.Dense(32, activation="relu")
        self.out = tf.keras.layers.Dense(1, name="price")

    def call(self, X):

        x_cat = self.embedding_block(X)

        x_num = tf.keras.layers.Concatenate()([
            tf.cast(X[c], tf.float32)
            for c in self.numerical_block.features
        ])

        x = tf.keras.layers.Concatenate()([x_cat, x_num])

        x = self.dense1(x)
        x = self.dense2(x)
        x = self.dense3(x)

        return self.out(x)

    
    def predict_embedding(self, X):

        X_ee = self.embedding_block(X)
 
        return X_ee

In [14]:
featured_data.model.unique

<bound method Series.unique of 2                camry
7                  crv
8                camry
9         town&country
10             corolla
              ...     
426866             crv
426870      expedition
426871      sierra1500
426877       silverado
426879         durango
Name: model, Length: 148421, dtype: str>

In [15]:
categorical_cols = [
    "type",
    "manufacturer",
    "model",
    "fuel",
    "transmission",
    "condition",
    "drive",
    "age_category",
    "usage_level",
    "posting_month",
    "posting_day",
    "day_of_week",
    # "is_weekend"
]


numerical_cols = [
    "year",
    "odometer",
    "age",
    # "odometer_per_age_buckets"
]

model = DNNModel(
    categorical_feature_vocabs={c: featured_data[c].unique().tolist() for c in categorical_cols},
    numerical_features=numerical_cols
)

# Train Embeddings

In [16]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(name="mae"),
    ],
)

In [17]:
train_inputs = {
    c: X_train[c].astype("string").to_numpy().reshape(-1, 1)
    for c in categorical_cols
}

train_inputs.update({
    c: X_train[c].to_numpy(dtype="float32").reshape(-1, 1)
    for c in numerical_cols
})

test_inputs = {
    c: X_test[c].astype("string").to_numpy().reshape(-1, 1)
    for c in categorical_cols
}

test_inputs.update({
    c: X_test[c].to_numpy(dtype="float32").reshape(-1, 1)
    for c in numerical_cols
})

y_train_ = y_train.to_numpy(dtype="float32")
y_test_ = y_test.to_numpy(dtype="float32")

In [18]:
for c in X_train.columns:
    n_missing = X_train[c].isna().sum()
    
    if n_missing > 0:
        print(f"{c}: {n_missing} missing")

In [19]:
history = model.fit(
    x=train_inputs,
    y=y_train_,
    validation_data=(test_inputs, y_test_),
    epochs=15,
    batch_size=256,
)

Epoch 1/15
464/464 [==============================] - 8s 3ms/step - loss: 150097984.0000 - mae: 8724.0322 - val_loss: 89354984.0000 - val_mae: 7276.7202
Epoch 2/15
464/464 [==============================] - 1s 2ms/step - loss: 82792224.0000 - mae: 6535.2251 - val_loss: 67872344.0000 - val_mae: 5983.1924
Epoch 3/15
464/464 [==============================] - 1s 2ms/step - loss: 55736948.0000 - mae: 5013.2451 - val_loss: 39547556.0000 - val_mae: 3978.4290
Epoch 4/15
464/464 [==============================] - 1s 2ms/step - loss: 39616748.0000 - mae: 4109.1230 - val_loss: 32887338.0000 - val_mae: 3658.6226
Epoch 5/15
464/464 [==============================] - 1s 2ms/step - loss: 33713200.0000 - mae: 3703.4736 - val_loss: 34168732.0000 - val_mae: 3714.6956
Epoch 6/15
464/464 [==============================] - 1s 2ms/step - loss: 30884860.0000 - mae: 3503.0557 - val_loss: 34022856.0000 - val_mae: 4005.9604
Epoch 7/15
464/464 [==============================] - 1s 2ms/step - loss: 29439324.0000

In [20]:
history.history.keys()

import plotly.graph_objects as go

def plot_learning_curve(loss, val_loss):
    
    epochs = list(range(1, len(loss) + 1))

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=epochs,
            y=loss,
            mode="lines",
            name="Train Loss"
        )
    )

    fig.add_trace(
        go.Scatter(
            x=epochs,
            y=val_loss,
            mode="lines",
            name="Validation Loss"
        )
    )

    fig.update_layout(
        title="Learning Curve",
        xaxis_title="Epoch",
        yaxis_title="Loss",
        template="plotly_white"
    )

    fig.show()

plot_learning_curve(
    history.history["mae"],
    history.history["val_mae"]
)

dict_keys(['loss', 'mae', 'val_loss', 'val_mae'])

# Inspecting Entity Embeddings

We start by inspecting the day_of_week entity embedding because it has only 2 dimensions (since $\lfloor\sqrt{7}\rfloor = 2$), making it the simplest to reason about.

Because the data comes from a listing service, we expect a weekend vs. weekday pattern: sellers tend to post more on weekends, and buyers tend to browse more on weekends. This behavioral difference should be reflected in the embedding — weekend days (Saturday, Sunday) should have higher dot-product similarity (**higher is the dot product higher is the similarity**) to each other than to weekdays


> PS: surprisingly Thursday is not that different from Sunday

In [21]:
day_of_week_ee = model.embedding_block.get_embedding_matrix("day_of_week")

day_of_week_ee

{'[UNK]': array([0.04807997, 0.04044969], dtype=float32),
 '4': array([-0.0237624 , -0.28909168], dtype=float32),
 '6': array([0.3674762 , 0.41797712], dtype=float32),
 '1': array([-0.17083196,  0.04020815], dtype=float32),
 '0': array([-0.4115051, -0.2058603], dtype=float32),
 '2': array([-0.326321 , -0.3083843], dtype=float32),
 '3': array([ 0.02791229, -0.05029681], dtype=float32),
 '5': array([0.08376966, 0.10246078], dtype=float32)}

In [24]:
day_of_week_ee.keys()

# NOTE: how onadas encodes
monday = day_of_week_ee['0']
tuesday = day_of_week_ee["1"]
wednesday = day_of_week_ee["2"]
thursday = day_of_week_ee["3"]
friday = day_of_week_ee["4"]
saturday = day_of_week_ee["5"]
sunday = day_of_week_ee["6"]

np.dot(sunday, monday)
np.dot(sunday, tuesday)
np.dot(sunday, wednesday)
np.dot(sunday, thursday)
np.dot(sunday, friday)
np.dot(sunday, saturday)

dict_keys(['[UNK]', '4', '6', '1', '0', '2', '3', '5'])

-0.23726323

-0.04597059

-0.24881278

-0.010765812

-0.12956582

0.07360962

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

# NOTE: build with help of LLM
def plot_sunday_similarity(day_of_week_ee):
    
    day_map = {
        "0": "Monday",
        "1": "Tuesday",
        "2": "Wednesday",
        "3": "Thursday",
        "4": "Friday",
        "5": "Saturday",
        "6": "Sunday",
    }

    sunday = day_of_week_ee["6"]

    rows = []
    for key, day_name in day_map.items():
        if key == "6":
            continue

        vec = day_of_week_ee[key]
        dot_sim = np.dot(sunday, vec)

        rows.append({
            "day": day_name,
            "dot_similarity_to_sunday": dot_sim,
        })

    df = pd.DataFrame(rows)

    fig = px.bar(
        df,
        x="day",
        y="dot_similarity_to_sunday",
        title="Dot-product similarity to Sunday embedding",
        labels={
            "day": "Day of week",
            "dot_similarity_to_sunday": "Dot similarity to Sunday",
        },
    )

    fig.update_layout(
        xaxis_categoryorder="array",
        xaxis_categoryarray=[
            "Monday", "Tuesday", "Wednesday",
            "Thursday", "Friday", "Saturday"
        ]
    )

    fig.show()

    return df

sunday_sim_df = plot_sunday_similarity(day_of_week_ee)
sunday_sim_df

,day,dot_similarity_to_sunday
0,Monday,-0.237263
1,Tuesday,-0.045971
2,Wednesday,-0.248813
3,Thursday,-0.010766
4,Friday,-0.129566
5,Saturday,0.073610


## Age as categorical values

Age is naturally a numerical feature, but we deliberately treat it as categorical and train an entity embedding for it. The hypothesis is that the model will learn a depreciation pattern from the price signal: a 1-year-old car should have an embedding closer to a 2-year-old car than to a 10-year-old car.

For capturing this, the similarity (cosine similarity) between the age-0 embedding and each other age embedding should decrease with age.

**In addition, we observe that the inverse of the similarity delta reflects the nonlinear depreciation pattern expected for car depreciation**.

In [35]:
age_category_ee = model.embedding_block.get_embedding_matrix("age_category")

age_category_ee.keys()

dict_keys(['[UNK]', '15', '7', '5', '13', '11', '10', '6', '9', '12', '17', 'old', '14', '18', '3', '19', '8', '4', '2', '1', '16', '0', '-1'])

In [29]:
age_zero = age_category_ee["0"]

# NOTE: 4 dim vector
age_zero

array([ 2.6014   , -2.5609553, -2.5639398,  2.5504642], dtype=float32)

In [30]:
age_zero = age_category_ee["0"]

age_zero.shape

ages = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12", "13", "14", "15", "16", "17", "18", "19", "old"]

cos_distances = {}
theta_distances = {}
norm_distances = {}
delta_distances = {}

for a in ages:

    norm_cos_distance = np.dot(age_zero, age_category_ee[a]) / (np.linalg.norm(age_zero) * np.linalg.norm(age_category_ee[a]))
    theta = np.arccos(norm_cos_distance)
    cos_distance = np.dot(age_zero, age_category_ee[a])
    delta_distance = np.linalg.norm(age_zero - age_category_ee[a])

    print(f"age: {a}, cos distance: {cos_distance}, angle (degrees): {np.degrees(theta)}; norm cos distance: {norm_cos_distance}; delta distance: {delta_distance}")

    cos_distances[a] = cos_distance
    theta_distances[a] = theta if not np.isnan(theta) else np.pi / 2.00
    norm_distances[a] = norm_cos_distance
    delta_distances[a] = delta_distance


(4,)

age: 0, cos distance: 26.404428482055664, angle (degrees): 0.01978234015405178; norm cos distance: 0.9999999403953552; delta distance: 0.0
age: 1, cos distance: 19.556503295898438, angle (degrees): 0.266878604888916; norm cos distance: 0.9999891519546509; delta distance: 1.332780122756958
age: 2, cos distance: 17.62506866455078, angle (degrees): 0.6723124384880066; norm cos distance: 0.9999311566352844; delta distance: 1.7090110778808594
age: 3, cos distance: 15.22579574584961, angle (degrees): 1.2392263412475586; norm cos distance: 0.9997661113739014; delta distance: 2.1764001846313477
age: 4, cos distance: 14.935379028320312, angle (degrees): 0.6221253871917725; norm cos distance: 0.9999410510063171; delta distance: 2.232196092605591
age: 5, cos distance: 12.6524019241333, angle (degrees): 1.4095879793167114; norm cos distance: 0.9996973872184753; delta distance: 2.676945686340332
age: 6, cos distance: 12.832816123962402, angle (degrees): 1.1700248718261719; norm cos distance: 0.9997

In [31]:
import plotly.graph_objects as go
import numpy as np

def plot_age_vs_distance(ages_cat, distances, distance_name: str = "Cosine Distance"):

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=ages_cat,
            y=distances,
            mode="lines+markers",
            name=distance_name
        )
    )

    # Force categorical ordering
    fig.update_layout(
        title=f"{distance_name} vs Age Category",
        xaxis=dict(
            title="Age Category",
            type="category",
            categoryorder="array",
            categoryarray=ages_cat
        ),
        yaxis=dict(
            title=distance_name
        ),
        template="plotly_white"
    )

    fig.show()

In [32]:
ages = [
    "0","1","2","3","4","5","6","7","8","9",
    "10","11","12","13","14","15","16","17",
    "18","19","old"
]

# Ensure correct order
distances = np.array([
    cos_distances[a] for a in ages
])

plot_age_vs_distance(ages, distances)

In [33]:
ages = [
    "0","1","2","3","4","5","6","7","8","9",
    "10","11","12","13","14","15","16","17",
    "18","19","old"
]

1.00/delta_distances["0"]

# Ensure correct order
distances = np.array([
    1.00/delta_distances[a] for a in ages
])


distances[0] = 1.00

plot_age_vs_distance(ages, distances, distance_name="Delta Distance")

/var/folders/tx/_wxs7hsj4tl4pkqn22jzb9yw0000gp/T/ipykernel_56633/573683222.py:7: RuntimeWarning: divide by zero encountered in divide
  1.00/delta_distances["0"]


inf

/var/folders/tx/_wxs7hsj4tl4pkqn22jzb9yw0000gp/T/ipykernel_56633/573683222.py:11: RuntimeWarning: divide by zero encountered in divide
  1.00/delta_distances[a] for a in ages


## Luxury vs normal vehicles clusters

Because the entity embeddings have higher dimensionality, we use **t-SNE** to project them into 2D for visualization while preserving local similarity structure.


The embedding shows a clear separation between **luxury** and **non-luxury** (mainstream/economy) vehicles. Brands such as Porsche, Mercedes-Benz, and Cadillac cluster together in a distinct region, indicating that the model has learned a consistent “luxury signal” from the price data. Mean while, the **economy and mainstream** brands (e.g., Nissan, Kia, Mitsubishi) form a separate cluster, suggesting lower price levels and similar market positioning.

**Premium brands** (e.g., BMW, Audi, Lexus) tend to lie between these two extremes, acting as a bridge between luxury and mainstream segments. This is expected, as they share characteristics of both groups.

In [43]:
train_data['manufacturer'].unique()

<ArrowStringArray>
[           'kia',         'nissan',      'chevrolet',           'ford',
        'hyundai',          'mazda',        'lincoln',          'rover',
       'chrysler',            'bmw',       'infiniti',           'jeep',
          'dodge',          'honda',         'subaru',         'toyota',
            'ram',   'mercedesbenz',          'acura',          'lexus',
            'gmc',           'audi',     'volkswagen',        'pontiac',
           'fiat',          'buick',       'cadillac',      'alfaromeo',
         'saturn',        'porsche',          'volvo',         'jaguar',
     'mitsubishi',          'tesla',           'mini',        'mercury',
 'harleydavidson',      'landrover',    'astonmartin',        'ferrari']
Length: 40, dtype: str

In [37]:
make_ee = model.embedding_block.get_embedding_matrix("manufacturer")
make_ee['kia'].shape

make_ee.keys()

(6,)

dict_keys(['[UNK]', 'toyota', 'honda', 'chrysler', 'ford', 'jeep', 'chevrolet', 'bmw', 'nissan', 'subaru', 'infiniti', 'gmc', 'dodge', 'volkswagen', 'lexus', 'mercedesbenz', 'porsche', 'acura', 'ram', 'hyundai', 'rover', 'cadillac', 'mini', 'tesla', 'mazda', 'mitsubishi', 'buick', 'jaguar', 'audi', 'fiat', 'kia', 'lincoln', 'volvo', 'mercury', 'pontiac', 'saturn', 'alfaromeo', 'harleydavidson', 'ferrari', 'landrover', 'astonmartin'])

In [38]:
bmw_emb = make_ee["bmw"]

bmw_emb.shape
bmw_emb

(6,)

array([-0.96807724, -0.92148745,  0.89814436,  0.98665583,  0.94146246,
        0.9162091 ], dtype=float32)

In [39]:
brand_tier = {
    "kia": "economy",
    "hyundai": "economy",
    "mitsubishi": "economy",
    "fiat": "economy",
    "saturn": "economy",

    "toyota": "mainstream",
    "honda": "mainstream",
    "nissan": "mainstream",
    "mazda": "mainstream",
    "subaru": "mainstream",
    "ford": "mainstream",
    "chevrolet": "mainstream",
    "dodge": "mainstream",
    "jeep": "mainstream",
    "ram": "mainstream",
    "gmc": "mainstream",
    "chrysler": "mainstream",
    "volkswagen": "mainstream",
    "buick": "mainstream",
    "pontiac": "mainstream",
    "mercury": "mainstream",

    "acura": "premium",
    "lexus": "premium",
    "infiniti": "premium",
    "lincoln": "premium",
    "cadillac": "premium",
    "volvo": "premium",
    "audi": "premium",
    "mini": "premium",
    "landrover": "premium",
    "rover": "premium",
    "tesla": "premium",
    "alfaromeo": "premium",

    "bmw": "luxury",
    "mercedesbenz": "luxury",
    "porsche": "luxury",
    "jaguar": "luxury",
    "astonmartin": "luxury",
    "ferrari": "luxury",
}

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA


def plot_embedding_2d_by_tier(
    embedding_dict,
    brand_tier,
    method="tsne",
    perplexity=10,
    random_state=42
):
    """
    Plot 2D visualization of embedding vectors colored by brand tier.

    Parameters
    ----------
    embedding_dict : dict[str, np.ndarray]
        Example: {"toyota": array(...), "bmw": array(...)}
    brand_tier : dict[str, str]
        Example: {"toyota": "mainstream", "bmw": "luxury"}
    method : str
        "tsne" or "pca"
    perplexity : int
        Used only for TSNE
    random_state : int
        Random seed
    """

    tier_levels = ["economy", "mainstream", "premium", "luxury"]

    labels = list(embedding_dict.keys())
    X = np.array([embedding_dict[k] for k in labels])

    if method.lower() == "tsne":
        n_samples = len(labels)
        safe_perplexity = min(perplexity, max(2, n_samples - 1))
        reducer = TSNE(
            n_components=2,
            perplexity=safe_perplexity,
            random_state=random_state,
            init="pca",
            learning_rate="auto"
        )
        X_2d = reducer.fit_transform(X)

    elif method.lower() == "pca":
        reducer = PCA(n_components=2, random_state=random_state)
        X_2d = reducer.fit_transform(X)

    else:
        raise ValueError("method must be 'tsne' or 'pca'")

    df_plot = pd.DataFrame({
        "make": labels,
        "x": X_2d[:, 0],
        "y": X_2d[:, 1],
    })

    df_plot["tier"] = df_plot["make"].map(brand_tier).fillna("unknown")
    df_plot["tier"] = pd.Categorical(
        df_plot["tier"],
        categories=tier_levels + ["unknown"],
        ordered=True
    )

    fig = px.scatter(
        df_plot,
        x="x",
        y="y",
        color="tier",
        hover_name="make",
        text="make",
        category_orders={"tier": tier_levels + ["unknown"]},
        title=f"2D Embedding Visualization by Tier ({method.upper()})"
    )

    fig.update_traces(marker=dict(size=10))
    fig.update_layout(template="plotly_white")
    fig.update_traces(textposition="top center")

    fig.show()

    return df_plot


In [41]:
df_plot = plot_embedding_2d_by_tier(
    embedding_dict=make_ee,
    brand_tier=brand_tier,
    method="tsne",
    perplexity=8
)